In [1]:
import torch
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from concurrent.futures import ProcessPoolExecutor
from collections import defaultdict

In [2]:
labels = torch.load('./beats/labels.pt')
data = pl.read_csv('ds/b05_onlykw_onlyexists.csv')

In [3]:
def get_frequencies(unique_labels, sample):
    counts = np.zeros_like(unique_labels)
    for i, v in enumerate(unique_labels):
        for w in sample:
            if v==w:
                counts[i] += 1
    return counts

In [4]:
labels = {k: v.numpy() for k, v in labels.items()}
unique_labels = set()
for v in labels.values():
    for w in v:
        unique_labels.add(int(w))
unique_labels=sorted(list(unique_labels))

In [5]:
new_labels = defaultdict(dict)
for k, v in labels.items():
    new_labels[k]['labels'] = v
    new_labels[k]['freq'] = get_frequencies(unique_labels, v)
    new_labels[k]['ecotype'] = data['Ecotype'][k]


In [6]:
X = [new_labels[k]['freq'] for k in new_labels.keys()]
y = [new_labels[k]['ecotype'] for k in new_labels.keys()]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [7]:
clf = LogisticRegression(random_state=59, max_iter=300).fit(X_train, y_train)

In [8]:
print(clf.score(X_train, y_train))
print(clf.score(X_test, y_test))

0.9969254419677172
0.9055588762701734


In [9]:
data_per_dataset=data.pivot(on="Ecotype", index="Dataset", values="Ecotype", aggregate_function="len")

In [10]:
less_imbalance = (      
      data_per_dataset
      .filter((pl.col("SRKW") > 0) & (pl.col("TKW") > 0))
      .with_columns((pl.col("SRKW") - pl.col("TKW")).abs().alias("diff"))
      .sort("diff")
      .head(1)
  )
print(less_imbalance["Dataset"])

most_transient = (
  data_per_dataset.sort("TKW", descending=True).head(10)
)
print(most_transient)

shape: (1,)
Series: 'Dataset' [str]
[
	"CarmanahPt"
]
shape: (10, 3)
┌───────────────┬──────┬──────┐
│ Dataset       ┆ SRKW ┆ TKW  │
│ ---           ┆ ---  ┆ ---  │
│ str           ┆ u32  ┆ u32  │
╞═══════════════╪══════╪══════╡
│ WVanIsl       ┆ 8    ┆ 1263 │
│ NorthBc       ┆ 0    ┆ 808  │
│ Cpe_Elz       ┆ 4    ┆ 530  │
│ BarkleyCanyon ┆ 23   ┆ 106  │
│ Quin_Can      ┆ 0    ┆ 61   │
│ CarmanahPt    ┆ 132  ┆ 55   │
│ StrGeoS1      ┆ 10   ┆ 47   │
│ StrGeoS2      ┆ 179  ┆ 17   │
│ SwanChan      ┆ 441  ┆ 7    │
│ orcasound_lab ┆ 160  ┆ 0    │
└───────────────┴──────┴──────┘


In [11]:
data_per_dataset.filter(pl.col("Dataset")=="CarmanahPt")

Dataset,SRKW,TKW
str,u32,u32
"""CarmanahPt""",132,55


In [14]:
X_train = [new_labels[k]['freq'] for k in new_labels.keys() if (data["Dataset"][k] != "CarmanahPt") & (data["Dataset"][k] != "Quin_Can")]
y_train = [new_labels[k]['ecotype'] for k in new_labels.keys() if (data["Dataset"][k] != "CarmanahPt") and (data["Dataset"][k] != "Quin_Can")]
y_train = [1 if y=='SRKW' else 0 for y in y_train]
X_test = [new_labels[k]['freq'] for k in new_labels.keys() if (data["Dataset"][k] == "CarmanahPt") | (data["Dataset"][k] == "Quin_Can")]
y_test = [new_labels[k]['ecotype'] for k in new_labels.keys() if (data["Dataset"][k] == "CarmanahPt") | (data["Dataset"][k] == "Quin_Can")]
y_test = [1 if y=='SRKW' else 0 for y in y_test]

In [58]:
clf = LogisticRegression(random_state=59, max_iter=300)
clf.fit(X_train, y_train)
print(clf.score(X_train, y_train))
print(clf.score(X_test, y_test))

0.9898648648648649
0.6693548387096774


In [67]:
pop_in_test = ((np.array(y_test)==1).sum(), (np.array(y_test)==0).sum())
print(f"A classifier that always predics SRKW would have an accuracy of {pop_in_test[0]/sum(pop_in_test)}")

A classifier that always predics SRKW would have an accuracy of 0.532258064516129


In [49]:
print(confusion_matrix(['SRKW' if x==1 else 'TKW' for x in y_test], ['SRKW' if x==1 else 'TKW' for x in clf.predict(X_test)], labels=['TKW', 'SRKW']))
print(f"TKW recall is {62/(54+62)}")
print(f"SRKW recall is {104/(104+28)}")


[[ 62  54]
 [ 28 104]]
TKW recall is 0.5344827586206896
SRKW recall is 0.7878787878787878


In [18]:
coefs = clf.coef_.squeeze()
top_coef = (int(coefs.argmax()), float(coefs[coefs.argmax()]))
min_coef = (int(coefs.argmin()), float(coefs[coefs.argmin()]))

In [19]:
top_coef
top_change = (np.e**top_coef[1] - 1)*100
print(f"{top_change:.2f}% change in odds")

386.22% change in odds


In [20]:
min_coef
min_change = (np.e**min_coef[1] - 1)*100
print(f"{min_change:.2f}% change in odds")


-82.65% change in odds


In [53]:
deciding_label = unique_labels[min_coef[0]]
i = 0
examples_idx = []
examples_lbs = []
#i = 0
for k, v in labels.items():
    if deciding_label in v:
        examples_idx.append(k)
        examples_lbs.append(v)
        i += 1
#    if i >=300:
        #break

examples = data[examples_idx][:]


In [54]:
examples

,...1,Soundfile,Dataset,LowFreqHz,HighFreqHz,FileEndSec,UTC,FileBeginSec,ClassSpecies,KW,KW_certain,Ecotype,Provider,AnnotationLevel,win_FilePath,FileOk,CallType,ID,CenterTime,Duration,CalltypeCategory,HasQ,CalltypeHasQ,EcotypeCertain,isPulsedCalls,Labels,HoldOut,Augmented,ShiftSec,SourceID,AugFromCT,pos_FilePath,gs_FilePath
i64,i64,str,str,f64,f64,f64,str,f64,str,i64,f64,str,str,str,str,bool,str,i64,f64,f64,str,bool,bool,bool,bool,str,str,bool,f64,i64,str,str,str
6017,159408,"""ST6249.220323090829.wav""","""CarmanahPt""",3000.0,3187.0,140.675,"""2022-03-23T09:10:49Z""",140.099,"""KW""",1,1.0,"""SRKW""","""DFO_WDA""","""call""","""E:/DCLDE/DFO_WDLP/Audio/Carman…",true,null,121471,140.387,0.576,null,false,false,true,false,"""SRKW""","""DCLDE_Train""",false,0.0,121471,null,"""dclde_2026_killer_whales/dfo_w…","""gs://noaa-passive-bioacoustic/…"
6089,160113,"""ST6249.220629205746.wav""","""CarmanahPt""",2085.0,2507.0,111.721,"""2022-06-29T20:59:36Z""",110.953,"""KW""",1,1.0,"""SRKW""","""DFO_WDA""","""call""","""E:/DCLDE/DFO_WDLP/Audio/Carman…",true,null,121976,111.337,0.768,null,false,false,true,false,"""SRKW""","""DCLDE_Train""",false,0.0,121976,null,"""dclde_2026_killer_whales/dfo_w…","""gs://noaa-passive-bioacoustic/…"
6091,160116,"""ST6249.220629205746.wav""","""CarmanahPt""",8039.0,8250.0,113.129,"""2022-06-29T20:59:38Z""",112.767,"""KW""",1,1.0,"""SRKW""","""DFO_WDA""","""call""","""E:/DCLDE/DFO_WDLP/Audio/Carman…",true,null,121979,112.948,0.362,null,false,false,true,false,"""SRKW""","""DCLDE_Train""",false,0.0,121979,null,"""dclde_2026_killer_whales/dfo_w…","""gs://noaa-passive-bioacoustic/…"
6117,160452,"""ST6249.220630000047.wav""","""CarmanahPt""",585.0,703.0,10.068,"""2022-06-30T00:00:56Z""",9.428,"""KW""",1,1.0,"""SRKW""","""DFO_WDA""","""call""","""E:/DCLDE/DFO_WDLP/Audio/Carman…",true,null,122226,9.748,0.64,null,false,false,true,false,"""SRKW""","""DCLDE_Train""",false,0.0,122226,null,"""dclde_2026_killer_whales/dfo_w…","""gs://noaa-passive-bioacoustic/…"
6479,163668,"""AMAR779.20210910T095609Z.wav""","""StrGeoS2""",2437.0,3468.0,182.569,"""2021-09-10T09:59:11Z""",182.1,"""KW""",1,1.0,"""SRKW""","""DFO_WDA""","""call""","""E:/DCLDE/DFO_WDLP/Audio/StrGeo…",true,null,124719,182.3345,0.469,null,false,false,true,false,"""SRKW""","""DCLDE_Train""",false,0.0,124719,null,"""dclde_2026_killer_whales/dfo_w…","""gs://noaa-passive-bioacoustic/…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
11903,44749,"""DFOCRP.KkHK0R2F-NML1.SM2M-3.20…","""NorthBc""",1000.0,1437.5,60.4982,"""2014-02-03T05:10:31Z""",60.0182,"""KW""",1,1.0,"""TKW""","""DFO_CRP""","""Call""","""E:/DCLDE/DFO_CRP/Audio/NorthBc…",true,"""T08""",151500,60.287,0.48,"""T08""",false,false,true,true,"""TKW""","""DCLDE_Train""",true,-0.0288,32924,"""T08""","""dclde_2026_killer_whales/dfo_c…","""gs://noaa-passive-bioacoustic/…"
11905,827,"""DFOCRP.KkHK0R2F-NML1.SM2M-3.20…","""NorthBc""",1750.0,2250.0,103.9414,"""2013-10-13T03:37:42Z""",102.5334,"""KW""",1,1.0,"""TKW""","""DFO_CRP""","""Call""","""E:/DCLDE/DFO_CRP/Audio/NorthBc…",true,"""T08""",151502,103.519,1.408,"""T08""",false,false,true,true,"""TKW""","""DCLDE_Train""",true,-0.2816,624,"""T08""","""dclde_2026_killer_whales/dfo_c…","""gs://noaa-passive-bioacoustic/…"
11913,827,"""DFOCRP.KkHK0R2F-NML1.SM2M-3.20…","""NorthBc""",1750.0,2250.0,103.96956,"""2013-10-13T03:37:42Z""",102.56156,"""KW""",1,1.0,"""TKW""","""DFO_CRP""","""Call""","""E:/DCLDE/DFO_CRP/Audio/NorthBc…",true,"""T08""",151510,103.519,1.408,"""T08""",false,false,true,true,"""TKW""","""DCLDE_Train""",true,-0.25344,624,"""T08""","""dclde_2026_killer_whales/dfo_c…","""gs://noaa-passive-bioacoustic/…"


In [55]:
print(examples.filter(pl.col("Ecotype")=="TKW").shape[0])
print(examples.filter(pl.col("Ecotype")=="SRKW").shape[0])

558
114


In [ ]:
from torchaudio.compliance import kaldi
from torchcodec.decoders import AudioDecoder
from IPython.display import Audio
import matplotlib.pyplot as plt
import os

root_audio = './ds/'


In [76]:
example = examples[4][:]
path = os.path.join(root_audio, example["pos_FilePath"].item())
song = (example["FileBeginSec"].item(), example["FileEndSec"].item())
soundwave = AudioDecoder(path, sample_rate=16000).get_samples_played_in_range(song[0], song[1]).data

Audio(data=soundwave, rate=16000)


In [ ]:
soundwave = soundwave * 2 ** 15       # scale to int16 range                                                                                                                                                                          
fbank = kaldi.fbank(soundwave, num_mel_bins=128, sample_frequency=16000, frame_length=25, frame_shift=10)                                                                                                                                      
# produces shape: (num_frames, 128)                                                                                                                                                                                                              
fbank = (fbank - fbank_mean) / (2 * fbank_std)    # normalize   fbank = fbank.unsqueeze(1)                        # (B, 1, T, 128) — treat as single-channel image                                                                                                                                               

NameError: name 'fbank_mean' is not defined